# 🐧 Chapter 4: Useful Linux Utilities

📖 Book: Chapter 4, pp. 150–188 · ⏱️ 2 hours · 🧱 Needs: Ch 3 · Level: Intermediate

---

> **🎯 Key idea:** Combine Linux diagnostics (`lsblk`, `parted`, `htop`, `strace`) with short Python helpers for JSON/CSV, TCP port checks, and load testing.

> **💡 Tip:** Run cells sequentially top-to-bottom (`Shift + Enter`).

> **⚠️ Warning:** Cells tagged `external` may require extra packages; cloud stacks incur costs if deployed.

## 🔄 Then vs. Now

> **📜 Then (book, ~2020):** `molotov` (minimally maintained)  
> **🆕 Now (2026):** `locust` / `hey` / `k6` (keep `ab` for quick checks)  
> **Why:** Active ecosystems & richer reporting

> **📜 Then (book, ~2020):** `os.urandom` + `base64`, `time.time()`  
> **🆕 Now (2026):** `secrets.token_urlsafe()`, `datetime.now(timezone.utc)`  
> **Why:** URL-safe tokens & timezone-aware UTC

---

## 4. 💡 Conceptual Cards

### 🚨 Real-World Problem
A database feels sluggish, and `df -h` says the disk isn't full—is the bottleneck throughput (MB/s) or IOPS? Or a service is bound only to `localhost` on a remote server behind a firewall. This chapter pairs Linux diagnostics (`lsblk`, `parted`, `ssh -L`, `strace`) with quick Python socket, disk, and load-test helpers.

### Concept: Disk & Process Diagnostics
**What:** `lsblk`, `parted`, `shutil.disk_usage`, `ps`  
**Why:** Distinguish IOPS vs MB/s & avoid 2 TB `fdisk` cap  
**How:** Query kernel/proc stats directly  
**When:** Troubleshooting slow I/O or full disks

📖 *Book listing:* `lsblk` & `shutil.disk_usage` Disk Audit (pp. 152–159)

**Plain-language analogy:** Throughput (MB/s) is how wide a highway is; IOPS is how many toll-booth cars pass per second. A database doing tiny random reads cares about toll-booth speed (IOPS), not highway width.

**Walkthrough:** The book contrasts `dd`/`iostat` throughput vs. IOPS and `parted` vs. `fdisk` (< 2 TB limit), then inspects block devices with `lsblk` without needing `sudo`.

### Concept: TCP Port & Load Checks
**What:** `socket.create_connection` & `locust`/`ab`  
**Why:** Verify SSH tunnels & connection pools  
**How:** Tests real TCP handshakes & concurrent HTTP  
**When:** Post-deploy smoke & stress tests

📖 *Book listing:* `molotov` Async Load Test (pp. 161–164)

**Plain-language analogy:** An SSH tunnel (`ssh -L`) is like a private underground pipe connecting your laptop's port directly to a locked server room behind a firewall.

**Walkthrough:** The book embeds short `python3 -c` snippets inside ZSH/Bash functions for CSV-to-JSON conversion and random tokens, and stresses HTTP endpoints with weighted async coroutines.

## 6. 🟢 Beginner Examples

#### Example 1: Beginner Example 1: Secure token & modern UTC Unix timestamp
*Expected output:* Printed labelled output below.

In [1]:
# Beginner Example 1: Secure token & modern UTC Unix timestamp
import secrets
from datetime import datetime, timezone

token = secrets.token_urlsafe(12)
epoch = int(datetime.now(timezone.utc).timestamp())
print('Secure token:', token, '| UTC epoch:', epoch)

Secure token: OfmaLtSOZPHq6FRe | UTC epoch: 1795770244


**What the output means:** The labelled `print(...)` lines confirm the transformed data or status returned by the logic above.

> **🧪 Try it:** Modify one input value above and re-run (`Shift + Enter`) to verify how the output changes.

#### Example 2: Beginner Example 2: CSV -> JSON one-liner equivalent
*Expected output:* Printed labelled output below.

In [2]:
# Beginner Example 2: CSV -> JSON one-liner equivalent
import csv, io, json

csv_buf = io.StringIO('host,port\nweb-01,80\nweb-02,443\n')
print('CSV -> JSON:', json.dumps(list(csv.DictReader(csv_buf))))

CSV -> JSON: [{"host": "web-01", "port": "80"}, {"host": "web-02", "port": "443"}]


**What the output means:** The labelled `print(...)` lines confirm the transformed data or status returned by the logic above.

> **🧪 Try it:** Modify one input value above and re-run (`Shift + Enter`) to verify how the output changes.

#### Example 3: Beginner Example 3: Disk usage & CPU load average
*Expected output:* Printed labelled output below.

In [3]:
# Beginner Example 3: Disk usage & CPU load average
import os, shutil

du = shutil.disk_usage('/')
print(f'Root free: {du.free / (1024**3):.2f} GiB | Load avg (1m, 5m, 15m): {os.getloadavg()}')

Root free: 207.08 GiB | Load avg (1m, 5m, 15m): (1.486328125, 1.640625, 1.80712890625)


**What the output means:** The labelled `print(...)` lines confirm the transformed data or status returned by the logic above.

> **🧪 Try it:** Modify one input value above and re-run (`Shift + Enter`) to verify how the output changes.

## 7. 🟡 Intermediate Examples: Process Inspection, Benchmarking & Load Testing

#### Example 4: Intermediate Example 1: Top CPU processes via subprocess + timeit comparison
*Expected output:* Printed labelled output below.

In [4]:
# Intermediate Example 1: Top CPU processes via subprocess + timeit comparison
import subprocess, timeit

cp = subprocess.run(['ps', '-eo', 'pid,pcpu,comm', '--sort=-pcpu'], capture_output=True, text=True, timeout=5)
print('\n'.join(cp.stdout.splitlines()[:5]))

t_list = min(timeit.repeat('sum([i for i in range(500)])', number=300, repeat=3))
t_gen = min(timeit.repeat('sum(i for i in range(500))', number=300, repeat=3))
print(f'timeit min: list={t_list:.5f}s | gen={t_gen:.5f}s')

    PID %CPU COMMAND
 341765 45.4 python
 284785 36.1 brave
 341569 33.9 jupyter-nbconve
 197023 25.0 brave
timeit min: list=0.00276s | gen=0.00460s


**What the output means:** The labelled `print(...)` lines confirm the transformed data or status returned by the logic above.

> **🧪 Try it:** Modify one input value above and re-run (`Shift + Enter`) to verify how the output changes.

#### Example 5: Intermediate Example 2 (External + Mock Fallback): Load testing (molotov / locust)
*Expected output:* Simulated fallback if optional package is missing.

In [5]:
# Intermediate Example 2 (External + Mock Fallback): Load testing (molotov / locust)
# Install: pip install locust  (or legacy book tool: pip install molotov)
try:
    import molotov
    print('molotov installed:', molotov.__version__)
except ImportError:
    import asyncio
    from concurrent.futures import ThreadPoolExecutor
    async def mock_load_test(n: int = 5):
        await asyncio.sleep(0.01)
        return {'requests': n, 'status_200': n}
    with ThreadPoolExecutor(1) as ex:
        res = ex.submit(asyncio.run, mock_load_test()).result()
    print('molotov not installed; simulated async load test:', res)

molotov not installed; simulated async load test: {'requests': 5, 'status_200': 5}


**What the output means:** If the optional library is installed, it runs live; otherwise the `except ImportError:` fallback prints a safe offline simulation.

> **🧪 Try it:** Install the optional package in your `.venv` (see the comment at top of the cell) and re-run to compare live vs. simulated output.

## 8. 🚀 Real-World DevOps Applications (Personal DevOps Learning Track)
*Supplementary extension:* **TCP Port Reachability Checker** to verify SSH tunnels or local containers.

#### Example 6: import socket
*Expected output:* Printed labelled output below.

In [6]:
import socket

def check_tcp_port(host: str, port: int, timeout: float = 0.5) -> bool:
    try:
        with socket.create_connection((host, port), timeout=timeout):
            return True
    except OSError:
        return False

print('Port 59999 reachable:', check_tcp_port('127.0.0.1', 59999))

Port 59999 reachable: False


**What the output means:** Shows the exact exception caught cleanly instead of crashing the whole script.

> **🧪 Try it:** Change the invalid input to a valid one and observe the `except` branch no longer triggers.

#### Example 7: Real Tool Demo: lsblk disk inspection (guarded with shutil.which)
*Expected output:* Printed labelled output below.

In [7]:
# Real Tool Demo: lsblk disk inspection (guarded with shutil.which)
import shutil, subprocess
if shutil.which('lsblk'):
    out = subprocess.run(['lsblk', '-d', '-o', 'NAME,SIZE,TYPE'], capture_output=True, text=True, check=True).stdout
    print('\n'.join(out.splitlines()[:5]))
else:
    print('lsblk not installed on host; skipped')

NAME      SIZE TYPE
loop0    23.8M loop
loop1    23.8M loop
loop2       4K loop
loop3    67.5M loop


**What the output means:** Checks whether the real CLI tool exists on `PATH`; if found, runs a safe read-only/dry-run check, otherwise skips cleanly.

> **🧪 Try it:** Try adding `--help` or inspecting `cp.returncode` to see how exit codes behave.

---

## 9. ⚠️ Common Mistakes & Troubleshooting

- ❌ **Using `fdisk` on >= 2 TB disks** -> Truncates or fails partition. ✅ Use `parted` with GPT.
- ❌ **Leaving `breakpoint()` in prod** -> Hangs headless cron/CI jobs. ✅ Remove breakpoints before commit.
- ❌ **Single `time.time()` benchmark** -> Noisy OS scheduling skew. ✅ Use `min(timeit.repeat(...))`.

#### Example 8: Troubleshooting: safe timeit without noisy outliers
*Expected output:* Printed labelled output below.

In [8]:
# Troubleshooting: safe timeit without noisy outliers
import timeit
runs = timeit.repeat('"-".join(["a", "b", "c"])', number=1000, repeat=3)
print('Best run:', round(min(runs), 6), 's')

Best run: 7.3e-05 s


**What the output means:** Shows the exact exception caught cleanly instead of crashing the whole script.

> **🧪 Try it:** Change the invalid input to a valid one and observe the `except` branch no longer triggers.

## 10. 🛠️ Hands-On Exercises
1. **(Beginner)** `gb_free(path: str = '/') -> float`: Return free GiB rounded to 2 decimals.
2. **(Beginner)** `generate_api_key(length: int = 24) -> str`: Use `secrets.token_urlsafe()`.
3. **(Beginner)** `csv_to_json(csv_text: str) -> str`: Convert CSV string to JSON list of dicts.
4. **(Intermediate)** `scan_ports(host: str, ports: list[int]) -> dict[int, bool]`: Check each port.
5. **(Intermediate)** `fastest_concat(n: int = 100) -> str`: Compare `join` vs `+` with `timeit` and return `'join'` or `'plus'`.

## 11. 🏆 Practical DevOps Challenge
`host_snapshot() -> dict`: Collect OS name, Python version, `/` disk used %, and localhost port checks `[22, 80, 443]`.

> **🧪 Try it:** The starter cell below reports `TODO` until you implement the function body.

> **🧪 Try it:** Run the starter cell below—it prints `TODO` until you implement the body.

#### 🧪 Starter Exercise: `Starter Exercise Cell (implement gb_free and run to verify!)`
*Expected output:* `TODO` message until implemented, then `✅ Starter checks PASSED!`.

In [9]:
# Starter Exercise Cell (implement gb_free and run to verify!)
def _run_starter_checks():
    assert gb_free('/') > 0
def gb_free(path: str = '/') -> float:
    raise NotImplementedError('Implement gb_free')

try:
    _run_starter_checks()
    print('✅ Starter checks PASSED!')
except NotImplementedError as e:
    print(f'🧪 TODO: {e}')

🧪 TODO: Implement gb_free


## 12. 🧠 Knowledge Checks (Inline Q&A for Practice)
**Q1:** What is the difference between IOPS and throughput?

<details>
<summary>✅ Answer 1</summary>

IOPS measures operations per second; throughput measures bytes per second (MB/s).

</details>

**Q2:** Why does `parted` replace `fdisk` on large disks?

<details>
<summary>✅ Answer 2</summary>

`fdisk` fails on disks >= 2 TB; `parted` supports GPT on large disks.

</details>

**Q3:** What does `ssh -L 9998:localhost:15672 user@host -N` do?

<details>
<summary>✅ Answer 3</summary>

Forwards local port 9998 to remote `localhost:15672` over encrypted SSH without a remote shell.

</details>

**Q4:** Why take `min()` of `timeit.repeat()`?

<details>
<summary>✅ Answer 4</summary>

`min()` reflects the fastest run with the least OS background noise.

</details>

**Q5:** What does `strace -f` reveal?

<details>
<summary>✅ Answer 5</summary>

All OS syscalls (`openat`, `stat`) made by the process and its children.

</details>

## 13. 🔑 Hints & Complete Solutions

<details>
<summary>💭 Hints (click to expand)</summary>

- **Ex 1**: `round(shutil.disk_usage(path).free / (1024 ** 3), 2)`.
- **Challenge**: Combine `platform.system()`, `shutil.disk_usage`, and `scan_ports`.

</details>

> **💡 Tip:** Complete solutions are in the collapsed/marked cell below.

<details>
<summary>✅ Complete Solutions (expand/run cell below to verify)</summary>

Run the verified solutions cell below to check all exercise & challenge assertions.

</details>

In [10]:
# Complete Solutions + Assert Verification
import csv, io, json, platform, secrets, shutil, timeit

def gb_free(path: str = '/') -> float:
    return round(shutil.disk_usage(path).free / (1024 ** 3), 2)

def generate_api_key(length: int = 24) -> str:
    return secrets.token_urlsafe(length)[:length]

def csv_to_json(csv_text: str) -> str:
    return json.dumps(list(csv.DictReader(io.StringIO(csv_text))))

def scan_ports(host: str, ports: list[int]) -> dict[int, bool]:
    return {p: check_tcp_port(host, p, timeout=0.1) for p in ports}

def host_snapshot() -> dict:
    du = shutil.disk_usage('/')
    return {
        'os': platform.system(),
        'python': platform.python_version(),
        'disk_used_pct': round((du.used / du.total) * 100, 1),
        'ports': scan_ports('127.0.0.1', [22, 80, 443]),
    }

assert gb_free() > 0
assert len(generate_api_key(24)) == 24
assert json.loads(csv_to_json('a,b\n1,2\n')) == [{'a': '1', 'b': '2'}]
snap = host_snapshot()
assert 'disk_used_pct' in snap and set(snap['ports']) == {22, 80, 443}
print('All Chapter 4 assertions PASSED!')

All Chapter 4 assertions PASSED!


---

## 14. 📌 Chapter Summary & Quick Reference

- `shutil.disk_usage('/')` — Cross-OS disk stats (e.g. `du.free / (1024**3)`)
- `socket.create_connection()` — Verify listening TCP port (e.g. `check_tcp_port('127.0.0.1', 80)`)
- `secrets.token_urlsafe(16)` — Cryptographic random token (e.g. `secrets.token_urlsafe(12)`)

### ✅ What You Can Now Do
- Distinguish disk throughput vs. IOPS and inspect block devices with `lsblk` and `shutil.disk_usage()`.
- Verify TCP port reachability with `socket.create_connection()` and benchmark snippets with `timeit.repeat()`.
- Generate secure tokens with `secrets` and UTC timestamps with `datetime.now(timezone.utc)`.

### ➡️ Next Chapter
Chapter 5 builds on this by packaging your Python CLI tools with `pyproject.toml` and running them as `systemd` services.